In [ ]:
# run classical ML model on ref-alt matrix

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.preprocessing import LabelEncoder


In [2]:

# --- PR Helper Functions ---
def compute_residue_scores_from_coef(coef, L):
    return np.abs(coef[:L])

def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    # catalog = catalog[
    #     (catalog["confidence"].isin(allowed_confidences)) &
    #     (catalog["intersectional"] == True)
    # ].copy()
    # catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) 
    return catalog


def evaluate_topk_precision_recall_refalt(gene_name, scores, catalog_df, k_vals=[10]):
    variants_df = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if variants_df.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return []

    total_actual_positives = len(np.unique(variants_df["aa_pos_0idx"]))
    print(f"Total confirmed resistance positions for {gene}: {total_actual_positives}")

    imp_df = pd.DataFrame({"Residue_Position": np.arange(len(scores)), "Importance": scores})
    imp_df_sorted = imp_df.sort_values("Importance", ascending=False)
    
    

    

    rows = []
    for k in k_vals:
        top_k = imp_df_sorted.head(k)
        # top_k = imp_df.nlargest(int(np.ceil(len(imp_df) * (k / 100))), "Importance")

        important_positions = set(top_k["Residue_Position"])

        true_positions = set(variants_df["aa_pos_0idx"])
        
        true_positives = len(true_positions.intersection(important_positions))
        total_predictions = len(important_positions)

        precision = true_positives / total_predictions if total_predictions > 0 else 0
        recall = true_positives / total_actual_positives if total_actual_positives > 0 else 0
        # f1 = 2 * prec * rec / (prec + rec + 1e-8)
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        assert precision <= 1.0, f"Precision > 1 for {gene_name} at k={k}"

        matched_df = variants_df[variants_df["aa_pos_0idx"].isin(important_positions)]

        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()
        
        print("Top residues from model:")
        print(top_k)

        print("Known resistance positions from WHO (1-indexed):")
        print(sorted(set(variants_df["aa_pos_0idx"])))


        rows.append({
            "gene": gene_name, "model": None, "k": k,
            "Total_Resistance_Positions": total_actual_positives,
            "TP": true_positives,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "identified_variants": ", ".join(identified_variants) if identified_variants else "None"
        })
    return rows


# --- Data Loaders ---
# def load_feature_matrix_and_labels(gene_name):
#     X = np.load(f"./data/feature_matrix_labels/{gene_name}_feature_matrix.npy")
#     y = np.load(f"./data/feature_matrix_labels/{gene_name}_labels.npy", allow_pickle=True)
#     return X, y


def load_feature_matrix_and_labels(gene_name):
    X = np.load(f"./data/feature_matrix_labels/{gene_name}_feature_matrix.npy")
    y = np.load(f"./data/feature_matrix_labels/{gene_name}_labels.npy", allow_pickle=True)

    # Drop mutation flag column (assumed to be first)
    X = X[:, 1:]
    print("data shape", X.shape)
    return X, y


def encode_labels(y):
    le = LabelEncoder()
    return le.fit_transform(y)

# --- Main Training + Evaluation ---
def run_models(gene_name):
    print(f"\n=== Running models for {gene_name} ===")
    X, y = load_feature_matrix_and_labels(gene_name)
    y_encoded = encode_labels(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
    )

    ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
    who_df = load_catalog("./data/filtered_variants_output.csv", ALLOWED_CONFIDENCES)

    # L = X.shape[1]  # sequence length in residue-level encoding
    L = X.shape[1]  # number of residues after dropping mutation flag

    results = {}
    pr_all_models = []

    # ----- Lasso -----
    lasso = LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100])
    lasso.fit(X_train, y_train)
    results['lasso'] = {
        'r2': lasso.score(X_test, y_test),
        'auc': roc_auc_score(y_test, lasso.predict(X_test)),
        'mse': mean_squared_error(y_test, lasso.predict(X_test))
    }
    # PR
    scores = compute_residue_scores_from_coef(lasso.coef_, L)
    
    # Save full scores
    model_name = "lasso"
    importance_df = pd.DataFrame({
        "Residue_Position": np.arange(L),
        "Importance": scores,
        "Model": model_name,
        "Gene": gene_name
    })
    importance_df.to_csv(f"./residue_importance/ref_alt/full_residue_scores_{gene_name}_{model_name}.csv", index=False)

    pr_rows = evaluate_topk_precision_recall_refalt(gene_name, scores, who_df)
    
    
    for r in pr_rows:
        r["model"] = "lasso"
    pr_all_models.extend(pr_rows)

    # ----- Ridge -----
    ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10])
    ridge.fit(X_train, y_train)
    results['ridge'] = {
        'r2': ridge.score(X_test, y_test),
        'auc': roc_auc_score(y_test, ridge.predict(X_test)),
        'mse': mean_squared_error(y_test, ridge.predict(X_test))
    }
    scores = compute_residue_scores_from_coef(ridge.coef_, L)
    pr_rows = evaluate_topk_precision_recall_refalt(gene_name, scores, who_df)
    for r in pr_rows:
        r["model"] = "ridge"
    pr_all_models.extend(pr_rows)

    # ----- Logistic Regression -----
    log_model = LogisticRegressionCV(
        cv=3, scoring="roc_auc", max_iter=5000,
        Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100],
        class_weight="balanced"
    )
    log_model.fit(X_train, y_train)
    results['logreg'] = {
        'r2': log_model.score(X_test, y_test),
        'auc': roc_auc_score(y_test, log_model.predict(X_test)),
        'mse': mean_squared_error(y_test, log_model.predict(X_test))
    }
    scores = compute_residue_scores_from_coef(log_model.coef_[0], L)  # note: coef_ shape is (1, L)
    pr_rows = evaluate_topk_precision_recall_refalt(gene_name, scores, who_df)
    for r in pr_rows:
        r["model"] = "logreg"
    pr_all_models.extend(pr_rows)

    # ----- Output -----
    for model_name, metrics in results.items():
        print(f"\n[{model_name.upper()}] Results for {gene_name}:")
        for metric, val in metrics.items():
            print(f"{metric}: {val:.4f}")

    # Save PR summary
    pr_df = pd.DataFrame(pr_all_models)
    pr_df.to_csv(f"./residue_importance/ref_alt/pr_summary_refalt_{gene_name}.csv", index=False)

    return results


In [3]:
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
gene_list=['rpoB','rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']


In [ ]:
all_results = {}
all_pr_rows = []
for gene in gene_list:
    try:
        print(f"\nRunning models for {gene}")
        result = run_models(gene)

        # Load the per-gene PR CSV just created
        pr_df = pd.read_csv(f"./residue_importance/ref_alt/pr_summary_refalt_{gene}.csv")
        all_pr_rows.append(pr_df)

        all_results[gene] = result
    except Exception as e:
        print(f"Error processing {gene}: {e}")

In [79]:
# Combine all per-gene PR summaries
all_pr_df = pd.concat(all_pr_rows, ignore_index=True)
all_pr_df

,gene,model,k,Total_Resistance_Positions,TP,precision,recall,F1,identified_variants
0,rpoB,lasso,10,26,8,0.8,0.307692,0.444444,"rpoB_p.Asp435Ala, rpoB_p.Gln432Asn, rpoB_p.His..."
1,rpoB,ridge,10,26,7,0.7,0.269231,0.388889,"rpoB_p.Asp435Ala, rpoB_p.Gln432Asn, rpoB_p.His..."
2,rpoB,logreg,10,26,7,0.7,0.269231,0.388889,"rpoB_p.Asn437Asp, rpoB_p.Asp435Ala, rpoB_p.Gln..."
3,rpsL,lasso,10,2,2,0.2,1.000000,0.333333,"rpsL_p.Lys43Arg, rpsL_p.Lys88Arg"
4,rpsL,ridge,10,2,2,0.2,1.000000,0.333333,"rpsL_p.Lys43Arg, rpsL_p.Lys88Arg"
5,rpsL,logreg,10,2,2,0.2,1.000000,0.333333,"rpsL_p.Lys43Arg, rpsL_p.Lys88Arg"
6,tlyA,lasso,10,2,0,0.0,0.000000,0.000000,NaN
7,tlyA,ridge,10,2,0,0.0,0.000000,0.000000,NaN
8,tlyA,logreg,10,2,0,0.0,0.000000,0.000000,NaN
9,pncA,lasso,10,95,8,0.8,0.084211,0.152381,"pncA_p.Cys14Arg, pncA_p.Gln10Arg, pncA_p.Gly97..."


In [80]:

all_pr_df.to_csv("residue_importance/refalt_k10.csv", index=False)
print("Combined PR summary saved to refalt_k10.csv")

Combined PR summary saved to refalt_k10.csv


In [81]:
# # Sort by 'gene' ascending and 'precision' descending
# df_sorted = all_pr_df.sort_values(by=['gene', 'precision'], ascending=[True, False])

# # Now, for each gene, pick the first row (highest precision per gene)
# best_per_gene = df_sorted.drop_duplicates(subset=['gene'], keep='first').reset_index(drop=True)

# print(best_per_gene)
# best_per_gene.to_csv("residue_importance/refalt_pr_overlap_summary.csv", index=False)

In [82]:
#Flatten it into a list of rows
rows = []
for gene, models in all_results.items():
    for model, metrics in models.items():
        row = {'gene': gene, 'model': model}
        row.update(metrics)
        rows.append(row)

In [83]:
# Convert to DataFrame
df = pd.DataFrame(rows)
df

,gene,model,r2,auc,mse
0,rpoB,lasso,0.818375,0.958284,0.032099
1,rpoB,ridge,0.834559,0.962801,0.029239
2,rpoB,logreg,0.962631,0.956946,0.031139
3,rpsL,lasso,0.435267,0.773858,0.120502
4,rpsL,ridge,0.435422,0.773858,0.120469
5,rpsL,logreg,0.773858,0.721115,0.173415
6,tlyA,lasso,-0.000001,0.500000,0.117260
7,tlyA,ridge,0.005227,0.502377,0.116647
8,tlyA,logreg,0.502377,0.505155,0.134266
9,pncA,lasso,0.144806,0.623855,0.094918


In [84]:
# Optional: rearrange columns
cols = ['gene', 'model'] + [c for c in df.columns if c not in ['gene', 'model']]
df = df[cols]

# View result
print(df.head())

   gene   model        r2       auc       mse
0  rpoB   lasso  0.818375  0.958284  0.032099
1  rpoB   ridge  0.834559  0.962801  0.029239
2  rpoB  logreg  0.962631  0.956946  0.031139
3  rpsL   lasso  0.435267  0.773858  0.120502
4  rpsL   ridge  0.435422  0.773858  0.120469


In [85]:
df.to_csv("auc_results/all_genes_regression_models_performance.csv", index=False)

## multiple genes

In [10]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, LogisticRegressionCV
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# --- Configuration ---
CATALOG_PATH = "./data/filtered_variants_output.csv"
SEQUENCE_INFO = "./data/catalog/protein_sequences.csv"
FEATURE_DIR = "./data/feature_matrix_labels"
PR_OUTPUT_DIR = "./residue_importance/ref_alt"

# --- Helper Functions ---
def compute_residue_scores_from_coef(coef, L):
    return np.abs(coef[:L])

def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[(catalog["confidence"].isin(allowed_confidences)) & (catalog["intersectional"] == True)].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    return catalog

def evaluate_topk_precision_recall_refalt(gene_name, scores, catalog_df, k_vals=[10]):
    variants_df = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if variants_df.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return []

    total_actual_positives = len(np.unique(variants_df["aa_pos_0idx"]))
    imp_df = pd.DataFrame({"Residue_Position": np.arange(len(scores)), "Importance": scores})
    imp_df_sorted = imp_df.sort_values("Importance", ascending=False)
    rows = []

    for k in k_vals:
        top_k = imp_df_sorted.head(k)
        important_positions = set(top_k["Residue_Position"])
        true_positions = set(variants_df["aa_pos_0idx"])
        true_positives = len(true_positions.intersection(important_positions))
        precision = true_positives / k
        recall = true_positives / total_actual_positives
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        matched_df = variants_df[variants_df["aa_pos_0idx"].isin(important_positions)]
        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()

        rows.append({
            "gene": gene_name, "model": None, "k": k,
            "Total_Resistance_Positions": total_actual_positives,
            "TP": true_positives,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "identified_variants": ", ".join(identified_variants) if identified_variants else "None"
        })
    return rows

def load_feature_matrix_and_labels(gene_name):
    X = np.load(f"{FEATURE_DIR}/{gene_name}_feature_matrix.npy")
    y = np.load(f"{FEATURE_DIR}/{gene_name}_labels.npy", allow_pickle=True)
    return X, y  # no mutation flag to drop

def encode_labels(y):
    return LabelEncoder().fit_transform(y)

# --- Main Function ---
def run_models_combined(multi_gene_name):
    print(f"\n=== Running models for {multi_gene_name} ===")
    X, y = load_feature_matrix_and_labels(multi_gene_name)
    y_encoded = encode_labels(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
    )

    who_df = load_catalog(CATALOG_PATH, ['1) Assoc w R', '2) Assoc w R - Interim'])

    # Reference protein lengths
    ref_seqs = pd.read_csv(SEQUENCE_INFO)
    gene_parts = multi_gene_name.split("_")
    gene_lengths = {
        gene: len(ref_seqs[ref_seqs["gene"] == gene]["protein_sequence"].values[0])
        for gene in gene_parts
    }

    # Compute start-end slices per gene
    offsets = {}
    offset = 0
    for g in gene_parts:
        offsets[g] = offset
        offset += gene_lengths[g]

    results = {}
    pr_all = []

    def model_runner(name, model, coef_extractor):
        model.fit(X_train, y_train)
        results[name] = {
            'r2': model.score(X_test, y_test),
            'auc': roc_auc_score(y_test, model.predict(X_test)),
            'mse': mean_squared_error(y_test, model.predict(X_test))
        }
        coefs = coef_extractor(model)
        for g in gene_parts:
            start = offsets[g]
            end = start + gene_lengths[g]
            scores = compute_residue_scores_from_coef(coefs, X.shape[1])[start:end]
            pr_rows = evaluate_topk_precision_recall_refalt(g, scores, who_df)
            for r in pr_rows:
                r["model"] = name
            pr_all.extend(pr_rows)

    model_runner("lasso", LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100]), lambda m: m.coef_)
    model_runner("ridge", RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10]), lambda m: m.coef_)
    model_runner("logreg", LogisticRegressionCV(cv=3, scoring="roc_auc", max_iter=5000,
        Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100], class_weight="balanced"), lambda m: m.coef_[0])

    os.makedirs(PR_OUTPUT_DIR, exist_ok=True)
    pd.DataFrame(pr_all).to_csv(f"{PR_OUTPUT_DIR}/pr_summary_refalt_{multi_gene_name}.csv", index=False)

    return results


In [11]:
results = run_models_combined("rpsL_gid")


=== Running models for rpsL_gid ===


## significance test

In [4]:
from sklearn.model_selection import StratifiedKFold
def run_models(gene_name):
    print(f"\\n=== Running models for {gene_name} ===")
    X, y = load_feature_matrix_and_labels(gene_name)
    y_encoded = encode_labels(y)

    ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
    who_df = load_catalog("./data/filtered_variants_output.csv", ALLOWED_CONFIDENCES)

    L = X.shape[1]  # number of residues after dropping mutation flag

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    all_fold_results = []
    all_pr_rows = []

    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X, y_encoded)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

        print(f"\\n[Fold {fold_idx + 1}]")

        # --- Model configurations ---
        model_configs = [
            ("lasso", LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100])),
            ("ridge", RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10])),
            ("logreg", LogisticRegressionCV(
                cv=3, scoring="roc_auc", max_iter=5000,
                Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100],
                class_weight="balanced"
            ))
        ]

        for model_name, model in model_configs:
            print(f"Training {model_name.upper()}...")
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            auc = roc_auc_score(y_test, preds)

            coef = model.coef_ if model_name != "logreg" else model.coef_[0]
            scores = compute_residue_scores_from_coef(coef, L)
            pr_rows = evaluate_topk_precision_recall_refalt(gene_name, scores, who_df)

            for r in pr_rows:
                r.update({
                    "model": model_name,
                    "fold": fold_idx + 1,
                    "auc": auc
                })
            all_pr_rows.extend(pr_rows)

            all_fold_results.append({
                "gene": gene_name,
                "model": model_name,
                "fold": fold_idx + 1,
                "auc": auc,
                "mse": mean_squared_error(y_test, preds),
                "r2": model.score(X_test, y_test)
            })

    # Save outputs
    auc_df = pd.DataFrame(all_fold_results)
 
    auc_df.to_csv(f"./auc_results/ref_alt/{gene_name}_cv_auc_per_fold.csv", index=False)

    pr_df = pd.DataFrame(all_pr_rows)
    pr_df.to_csv(f"./residue_importance/ref_alt/{gene_name}_cv_pr_per_fold.csv", index=False)

    print(f"Saved fold-wise AUC and interpretability results for {gene_name}.")
    return auc_df


In [7]:
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
gene_list=['rpoB','tlyA','pncA','eis','katG','inhA','gyrB', 'gyrA']


In [ ]:
all_results = {}
all_pr_rows = []
for gene in gene_list:
    try:
        print(f"\nRunning models for {gene}")
        result = run_models(gene)

        # Load the per-gene PR CSV just created
        pr_df = pd.read_csv(f"./residue_importance/ref_alt/{gene_name}_cv_pr_per_fold.csv")
        all_pr_rows.append(pr_df)

        all_results[gene] = result
    except Exception as e:
        print(f"Error processing {gene}: {e}")

In [21]:
multi_gene_drugs = ["katG_inhA", "rpsL_gid", "embA_embB_embC", "gyrA_gyrB", "ethA_ethR"]


In [22]:
PR_OUTPUT_DIR = "./residue_importance/ref_alt"

In [23]:
from sklearn.model_selection import StratifiedKFold
def evaluate_topk_precision_recall_refalt(gene_name, scores, catalog_df, k_vals=[10], offset=0):
    variants_df = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if variants_df.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return []

    total_actual_positives = len(np.unique(variants_df["aa_pos_0idx"]))
    print(f"Total confirmed resistance positions for {gene_name}: {total_actual_positives}")

    # Use global Residue_Position via offset
    imp_df = pd.DataFrame({
        "Residue_Position": np.arange(len(scores)) + offset,
        "Importance": scores
    })
    imp_df_sorted = imp_df.sort_values("Importance", ascending=False)

    rows = []
    for k in k_vals:
        top_k = imp_df_sorted.head(k)
        important_positions = set(top_k["Residue_Position"])
        true_positions = set(variants_df["aa_pos_0idx"])
        true_positives = len(true_positions.intersection(important_positions))

        precision = true_positives / k
        recall = true_positives / total_actual_positives if total_actual_positives > 0 else 0
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        matched_df = variants_df[variants_df["aa_pos_0idx"].isin(important_positions)]
        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()

        print("Top residues from model:")
        print(top_k)
        print("Known resistance positions from WHO (1-indexed):")
        print(sorted(set(variants_df["aa_pos_0idx"])))

        rows.append({
            "gene": gene_name, "model": None, "k": k,
            "Total_Resistance_Positions": total_actual_positives,
            "TP": true_positives,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "identified_variants": ", ".join(identified_variants) if identified_variants else "None"
        })

    return rows



def load_feature_matrix_and_labels(gene_name):
    X = np.load(f"./data/feature_matrix_labels/{gene_name}_feature_matrix.npy")
    y = np.load(f"./data/feature_matrix_labels/{gene_name}_labels.npy", allow_pickle=True)


    print("data shape", X.shape)
    return X, y

def run_models_combined(multi_gene_name, n_splits=5):
    print(f"\n=== Running CV models for {multi_gene_name} ===")

    X, y = load_feature_matrix_and_labels(multi_gene_name)
    y_encoded = encode_labels(y)

    who_df = load_catalog("./data/filtered_variants_output.csv", ['1) Assoc w R', '2) Assoc w R - Interim'])
    ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')

    gene_parts = multi_gene_name.split("_")
    gene_lengths = {
        gene: len(ref_seqs[ref_seqs["gene"] == gene]["protein_sequence"].values[0])
        for gene in gene_parts
    }

    # Compute coefficient slices per gene
    offsets = {}
    offset = 0
    for g in gene_parts:
        offsets[g] = offset
        offset += gene_lengths[g]

    all_auc_rows = []
    all_pr_rows = []

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    model_configs = [
        ("lasso", LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100]), lambda m: m.coef_),
        ("ridge", RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10]), lambda m: m.coef_),
        ("logreg", LogisticRegressionCV(cv=3, scoring="roc_auc", max_iter=5000,
            Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100], class_weight="balanced"), lambda m: m.coef_[0])
    ]

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y_encoded)):
        print(f"\n[Fold {fold_idx + 1}]")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

        for model_name, model, coef_extractor in model_configs:
            print(f"Training {model_name.upper()}...")
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            auc = roc_auc_score(y_test, preds)

            all_auc_rows.append({
                "gene": multi_gene_name,
                "model": model_name,
                "fold": fold_idx + 1,
                "auc": auc,
                "mse": mean_squared_error(y_test, preds),
                "r2": model.score(X_test, y_test)
            })

            coefs = coef_extractor(model)

            # Compute PR separately for each gene in the set
            for g in gene_parts:
                start, end = offsets[g], offsets[g] + gene_lengths[g]
                print(f"[{multi_gene_name} | {g}] Offset: {start}–{end} | Length: {end - start} | Expected: {gene_lengths[g]}")

                scores = compute_residue_scores_from_coef(coefs, X.shape[1])[start:end]

                pr_rows = evaluate_topk_precision_recall_refalt(g, scores, who_df, offset=start)
                for r in pr_rows:
                    r.update({
                        "model": model_name,
                        "fold": fold_idx + 1,
                        "gene_set": multi_gene_name
                    })
                all_pr_rows.extend(pr_rows)

    # Save outputs
    os.makedirs(PR_OUTPUT_DIR, exist_ok=True)
    auc_df = pd.DataFrame(all_auc_rows)
    auc_df.to_csv(f"./auc_results/ref_alt/{multi_gene_name}_cv_auc_per_fold.csv", index=False)

    pr_df = pd.DataFrame(all_pr_rows)
    pr_df.to_csv(f"./residue_importance/ref_alt/{multi_gene_name}_cv_pr_per_fold.csv", index=False)

    print(f"Saved CV AUC and PR results for {multi_gene_name}")
    return auc_df


In [ ]:
for gene in multi_gene_drugs:
    try:
        print(f"\nRunning models for {gene}")
        result = run_models_combined(gene)  # Already does CV + interpretability + saves CSV
    except Exception as e:
        print(f"Error processing {gene}: {e}")
